In [0]:
import pandas as pd

catalogs = ["gold_dev", "silver_dev"]

In [0]:
%sql
SELECT DISTINCT
  table_catalog,
  table_schema,
  table_name
FROM 
  system.information_schema.columns
WHERE  
  table_schema NOT IN ('access', 'default', 'information_schema')
ORDER BY 2, 3, 1

In [0]:
%sql
SELECT 
  table_catalog AS `catalog`,
  table_schema AS `schema`,
  table_name,
  column_name,
  ordinal_position AS `column_ordinal_position`,
  full_data_type AS `column_data_type`
FROM 
  system.information_schema.columns
WHERE
  table_schema NOT IN ('access', 'default', 'information_schema')
ORDER BY
  table_catalog,
  table_schema,
  table_name,
  ordinal_position

In [0]:
cols_df = _sqldf.toPandas()
cols_df[cols_df["catalog"].isin(catalogs)]
cols_df

In [0]:
%sql
SELECT
  kcu.*
FROM
  system.information_schema.table_constraints tc
JOIN
  system.information_schema.key_column_usage kcu
ON
  tc.constraint_name = kcu.constraint_name
  AND tc.table_schema = kcu.table_schema
  AND tc.table_name = kcu.table_name
  AND tc.table_catalog = kcu.table_catalog
WHERE
  tc.constraint_type = 'PRIMARY KEY'
ORDER BY kcu.constraint_catalog, kcu.constraint_schema, kcu.constraint_name

In [0]:
pk_df = _sqldf.toPandas()
pk_df[pk_df["constraint_catalog"].isin(catalogs)]

# build the map
pk_map = (
    pk_df
    .groupby(["table_catalog", "table_schema", "table_name"])["column_name"]
    .apply(list)
    .reset_index()
)

# convert to dict with desired key format
pk_map = {
    f"{row.table_catalog}.{row.table_schema}.{row.table_name}": row.column_name
    for _, row in pk_map.iterrows()
}
pk_map

In [0]:
%sql
SELECT
  -- Child (FK side)
  fk_kcu.constraint_catalog      AS fk_constraint_catalog,
  fk_kcu.constraint_schema       AS fk_constraint_schema,
  fk_kcu.constraint_name         AS fk_constraint_name,
  fk_kcu.table_catalog           AS fk_table_catalog,
  fk_kcu.table_schema            AS fk_table_schema,
  fk_kcu.table_name              AS fk_table_name,
  fk_kcu.column_name             AS fk_column_name,
  fk_kcu.ordinal_position        AS fk_ordinal_position,

  -- Parent (PK side)
  pk_kcu.table_catalog           AS pk_table_catalog,
  pk_kcu.table_schema            AS pk_table_schema,
  pk_kcu.table_name              AS pk_table_name,
  pk_kcu.column_name             AS pk_column_name

FROM system.information_schema.referential_constraints rc

JOIN system.information_schema.key_column_usage fk_kcu
  ON rc.constraint_name = fk_kcu.constraint_name
  AND rc.constraint_schema = fk_kcu.constraint_schema

JOIN system.information_schema.key_column_usage pk_kcu
  ON rc.unique_constraint_name = pk_kcu.constraint_name
  AND rc.unique_constraint_schema = pk_kcu.constraint_schema
  AND fk_kcu.ordinal_position = pk_kcu.ordinal_position

ORDER BY
  fk_kcu.constraint_catalog,
  fk_kcu.constraint_schema,
  fk_kcu.constraint_name,
  fk_kcu.ordinal_position;

In [0]:
from collections import defaultdict
fk_df = _sqldf.toPandas()
fk_df[fk_df["pk_table_catalog"].isin(catalogs)]

fk_list = []
for _, row in fk_df.iterrows():
    key = f"{row.pk_table_catalog}.{row.fk_table_schema}.{row.fk_table_name}"
    
    fk_list.append({
        "pk_table_catalog": row.pk_table_catalog,
        "pk_table_schema": row.pk_table_schema,
        "pk_table_name": row.pk_table_name,
        "pk_column_name": row.pk_column_name,

        "fk_table_catalog": row.fk_table_catalog,
        "fk_table_schema": row.fk_table_schema,
        "fk_table_name": row.fk_table_name,
        "fk_column_name": row.fk_column_name,
    })
fk_list

In [0]:
key_columns_list = []

for k, v in pk_map.items():
    for col in v:
        key_columns_list.append(f"{k}.{col}")

for rel in fk_list:
    key_columns_list.append(
        f"{rel['fk_table_catalog']}.{rel['fk_table_schema']}.{rel['fk_table_name']}.{rel['fk_column_name']}"
    )
    
    key_columns_list.append(
        f"{rel['pk_table_catalog']}.{rel['pk_table_schema']}.{rel['pk_table_name']}.{rel['pk_column_name']}"
    )

key_columns_set = set(key_columns_list)
key_columns_set

In [0]:

hide_non_key_columns = True

# Optional: sort to ensure correct column order
cols_df = cols_df.sort_values(
    by=["catalog", "schema", "table_name", "column_ordinal_position"]
)
cols_df = cols_df[cols_df["catalog"].isin(catalogs)]

dbml_lines = []

for (catalog, schema, table), group in cols_df.groupby(["catalog","schema", "table_name"]):
    table_fq = f"{catalog}__{schema}.{table}"
    dbml_lines.append(f"Table {table_fq}" + "{")
    
    pk_columns = pk_map.get(table_fq, [])
    #print(f"{table_fq} has primary keys: {pk_columns}")
    for _, row in group.iterrows():

        # For now, force lower column name because it's not updating NPI column names
        col_name = row["column_name"].lower()
        col_type = row["column_data_type"]
        
        if col_name in pk_columns:
            dbml_lines.append(f"  {col_name} {col_type} [primary key]")
        elif hide_non_key_columns is False or f"{catalog}.{schema}.{table}.{col_name}" in key_columns_set:
            dbml_lines.append(f"  {col_name} {col_type}")

    dbml_lines.append("}\n")


# Add foreign key references
for key_relationships in fk_list:
    pk_table_catalog = key_relationships["pk_table_catalog"]
    pk_table_schema = key_relationships["pk_table_schema"]
    pk_table_name = key_relationships["pk_table_name"]
    pk_column_name = key_relationships["pk_column_name"]

    fk_table_catalog = key_relationships["fk_table_catalog"]
    fk_table_schema = key_relationships["fk_table_schema"]
    fk_table_name = key_relationships["fk_table_name"]
    fk_column_name = key_relationships["fk_column_name"]

    dbml_lines.append(f"Ref: {pk_table_catalog}__{pk_table_schema}.{pk_table_name}.{pk_column_name} > {fk_table_catalog}__{fk_table_schema}.{fk_table_name}.{fk_column_name}")

dbml_output = "\n".join(dbml_lines)

print(dbml_output)

# Future Addition: Pull primary keys from the tables

In [0]:
%sql
SELECT *
FROM
  system.information_schema.table_constraints tc
WHERE
  constraint_type = 'PRIMARY KEY'

In [0]:
%sql
SELECT *
FROM
  system.information_schema.key_column_usage kcu

In [0]:
%sql
SELECT
  kcu.*
FROM
  system.information_schema.table_constraints tc
JOIN
  system.information_schema.key_column_usage kcu
ON
  tc.constraint_name = kcu.constraint_name
  AND tc.table_schema = kcu.table_schema
  AND tc.table_name = kcu.table_name
WHERE
  tc.constraint_type = 'PRIMARY KEY'

In [0]:
%sql
SELECT *
FROM
  system.information_schema.key_column_usage kcu
ORDER BY
  1, 2, 3